# 🤖 AI Chatbot using Transformers

Welcome to the AI-powered conversational chatbot built using Hugging Face Transformers.

This chatbot uses a **hybrid approach**:
- Rule-based intent matching for accurate responses
- Transformer model for natural conversation




**Type `exit` to end the conversation.**

In [38]:
# pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import re

# ------------------ DEVICE ------------------
device = "cuda" if torch.cuda.is_available() else "cpu"


In [31]:
# ------------------ MODEL ------------------
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

In [32]:
# ------------------ CONFIG ------------------
MAX_HISTORY = 5   # keep last 5 exchanges only

# ------------------ KNOWLEDGE BASE ------------------
knowledge_base = {
    "greeting": {
        "patterns": ["hi", "hello", "hey"],
        "response": "Hello! Nice to meet you. How can I assist you today?"
    },
    "ai": {
        "patterns": ["artificial intelligence", "ai"],
        "response": "Artificial Intelligence refers to machines that simulate human intelligence like learning, reasoning, and decision-making."
    },
    "python": {
        "patterns": ["who created python", "python creator"],
        "response": "Python was created by Guido van Rossum in 1991."
    }
}

In [33]:
# ------------------ INTENT MATCHING ------------------
def match_intent(user_input):
    user_input = user_input.lower()

    for intent in knowledge_base.values():
        for pattern in intent["patterns"]:
            if pattern in user_input:
                return intent["response"]

    return None


# ------------------ CLEAN RESPONSE ------------------
def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = text.strip()
    return text

In [34]:
# ------------------ GENERATE ------------------
def generate_response(user_input, chat_history_ids, history_text):

    # 🔹 Intent-based response (HIGH PRIORITY)
    intent_response = match_intent(user_input)
    if intent_response:
        return intent_response, chat_history_ids, history_text

    # 🔹 Keep only last N history
    history_text = history_text[-MAX_HISTORY:]

    # 🔹 Build structured prompt (better than raw concat)
    prompt = ""
    for h in history_text:
        prompt += f"User: {h['user']}\nBot: {h['bot']}\n"
    prompt += f"User: {user_input}\nBot:"

    new_input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt').to(device)

    # 🔹 Generate
    output_ids = model.generate(
        new_input_ids,
        max_length=300,
        pad_token_id=tokenizer.eos_token_id,

        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,

        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )

    # 🔹 Decode only new response
    response = tokenizer.decode(output_ids[:, new_input_ids.shape[-1]:][0], skip_special_tokens=True)

    response = clean_text(response)

    # 🔹 Fallback if garbage
    if len(response) < 2:
        response = "I'm not sure about that. Can you please rephrase?"

    # 🔹 Update memory
    history_text.append({
        "user": user_input,
        "bot": response
    })

    return response, output_ids, history_text



In [37]:

# ------------------ CHAT LOOP ------------------
print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

chat_history_ids = None
history_text = []

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    response, chat_history_ids, history_text = generate_response(
        user_input,
        chat_history_ids,
        history_text
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. How can I help you today?

User: Hello
Chatbot: Hello! Nice to meet you. How can I assist you today?
User: Who created Python?
Chatbot: Python was created by Guido van Rossum in 1991.
User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to machines that simulate human intelligence like learning, reasoning, and decision-making.
User: exit
Chatbot: Goodbye!


# 👋 Thank You

Thank you for interacting with the AI Chatbot.

This project highlights the power of transformer-based models in building intelligent conversational systems.
